# 01 — データ探索・EDA・検証

保存済みデータを読み込み、品質・分布・分割の状態を確認します。
Notebookは設定・読込みAPIの呼出し・分析表の表示・可視化に使い、処理は `src/music_recsys/` に置きます。
`02_recommendation.ipynb` は推薦結果、`03_analysis.ipynb` は評価指標・比較を扱います。

## 1. Overview

| 対象 | 内容・規模 |
| --- | --- |
| PostgreSQLの保存済みカタログ | Music4All-Onion / 配布ラベルv2。公開済みsnapshotを指定すると100曲・100ユーザーを読む |
| DB未接続の既定モード | Gitに保存した合成catalog fixture。4曲・2ユーザー。実DBや実Music4Allのサンプルではない |
| 参考資料 | Music4All-Onion全量イベントの保存済み集計。カタログsnapshotとは別の対象規模 |

Music4Allの出典は [Zenodo record 15394646](https://zenodo.org/records/15394646)。
100曲・100ユーザーsnapshotに再生関係・ユーザー属性はなく、正式なTrain / Validation / Testは未作成です。

既定の実行はDB・配布ファイル・常駐MLflowを必要としません。
`uv run --locked --group data --group notebook --group tracking jupyter lab` で開き、カーネルを再起動して上から実行します。
DBを読む場合は `.env` を起動時に渡します。接続値をセルへ書かないでください。

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from uuid import UUID

import matplotlib.pyplot as plt
from IPython.display import display
from pandas import DataFrame

from music_recsys.analysis import split_candidate_tables, temporal_rows
from music_recsys.snapshot_analysis import (
    load_fixture,
    load_snapshot_frames,
    snapshot_eda,
    validation_rows,
)

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)
SNAPSHOT_ID = None  # 公開済みsnapshotのUUID文字列。NoneならDBに接続しない。
DATASET_KEY = "music4all-onion"
VERSION_LABEL = "v2"
FIXTURE_PATH = ROOT / "examples/catalog/snapshot.json"
PROFILE_PATH = Path(
    os.environ.get("MUSIC_RECSYS_PROFILE", str(ROOT / "experiments/music4all/eda/profile.json"))
)

mode = "PostgreSQL saved snapshot" if SNAPSHOT_ID else "Saved synthetic fixture (not the database)"
display(
    {
        "mode": mode,
        "dataset": DATASET_KEY if SNAPSHOT_ID else "synthetic-catalog",
        "version": VERSION_LABEL if SNAPSHOT_ID else "fixture-v1",
        "snapshot_id": SNAPSHOT_ID,
        "source": "https://zenodo.org/records/15394646"
        if SNAPSHOT_ID
        else "examples/catalog/snapshot.json",
        "expected_tracks": 100 if SNAPSHOT_ID else 4,
        "expected_users": 100 if SNAPSHOT_ID else 2,
    }
)

## 2. Data Loading

PostgreSQLモードでは `load_snapshot_frames` が指定した公開済みsnapshotに限定してSQLを実行し、楽曲・ユーザーをDataFrameへ読み込みます。
SQLは読み取り専用トランザクションで実行し、snapshotに結び付いた不変の `dataset_version_id` と配布ラベルを保持・照合します。
最新のsnapshotを暗黙に選択しません。Parquetは既存の `load_snapshot` APIで検証します。

DB未接続モードでは保存済み合成fixtureをDataFrameへ読みます。以下の表示は規模・カラム・出典に限り、ユーザーID・生レコードを出力しません。
全量イベントは加工データ本体を読まず、保存済みEDAのJSONだけを読みます。


In [ ]:
if SNAPSHOT_ID:
    from music_recsys.ingestion.postgres import engine_from_environment

    engine = engine_from_environment()
    try:
        snapshot = load_snapshot_frames(
            engine,
            ROOT / "data/imports",
            UUID(SNAPSHOT_ID),
            dataset_key=DATASET_KEY,
            version_label=VERSION_LABEL,
        )
    finally:
        engine.dispose()
else:
    snapshot = load_fixture(FIXTURE_PATH)

tracks_df, users_df = snapshot.tracks, snapshot.users
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
display(snapshot.overview)
display(
    {
        "tracks_frame": {"rows": len(tracks_df), "columns": list(tracks_df.columns)},
        "users_frame": {"rows": len(users_df), "columns": list(users_df.columns)},
        "reference_eda": {
            key: profile[key] for key in ("n_events", "n_users", "n_items", "min_time", "max_time")
        },
    }
)

## 3. Data Validation

件数、元ID・内部UUIDの欠損／重複、UUIDの形式、タグ未取得と空タグを確認します。
`payload=NULL` はタグ未取得、`payload.tags={}` は取得済み・タグなしです。欠損を空タグへ変換しません。

実DBモードは所属・メタデータ参照・作成run・公開状態・manifest／fingerprintを検証し、Parquetのhash・schema・全行内容も照合します。
検証失敗は読込みAPIが拒否します。fixtureモードのDB／Parquet検証は `not checked` と表示し、合成データを実DBの検証結果として扱いません。
`expected=None` の欠損タグ数・空タグ数は観測値で、その存在自体をエラーにしません。

In [ ]:
validation = validation_rows(
    snapshot,
    expected_tracks=100 if SNAPSHOT_ID else 4,
    expected_users=100 if SNAPSHOT_ID else 2,
)
display(DataFrame(validation))

## 4. EDA

この章が主な探索です。曲タグの有無、タグ出現率、曲ごとのタグ数、重み分布を見ます。
タグ出現率の分母は読込んだカタログの全曲数です。曲ごとのタグ数はタグ取得済みの曲だけを数え、空タグは0として含めます。
重みは配布値（1〜100）で、嗜好確率や推薦scoreではありません。

fixtureモードの図は合成データの例示です。実Music4Allの分布と解釈しません。
ユーザーにはIDしか保存していないため、年齢・性別・地域・嗜好などの属性分布は分析できません。

In [ ]:
eda = snapshot_eda(snapshot)
display(eda["tag_coverage"])
display(eda["tags"].head(15))
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
coverage = eda["tag_coverage"]
axes[0, 0].bar(coverage["status"], coverage["n_tracks"], color=["#8b507b", "#c77936", "#256b78"])
axes[0, 0].set(title="Tag availability", ylabel="Tracks")
top_tags = eda["tags"].head(10)
axes[0, 1].barh(top_tags["tag"], top_tags["track_fraction"], color="#256b78")
axes[0, 1].invert_yaxis()
axes[0, 1].set(title="Tag occurrence among all tracks", xlabel="Track fraction", xlim=(0, 1))
axes[1, 0].hist(eda["tag_counts"], bins="auto", color="#256b78", edgecolor="white")
axes[1, 0].set(title="Tags per track (known metadata)", xlabel="Tag count", ylabel="Tracks")
axes[1, 1].hist(eda["tag_weights"], bins=10, range=(1, 101), color="#c77936", edgecolor="white")
axes[1, 1].set(title="Supplied tag weights", xlabel="Weight", ylabel="Tag assignments")
fig.suptitle(f"Catalog EDA: {snapshot.overview['dataset_key']} / {snapshot.mode}")
fig.tight_layout()
plt.show()

### 参考：全量イベントの時刻分布

以下は保存済み全量集計JSONです。上のカタログsnapshot・合成fixtureとは対象が異なります。
配布時刻のタイムゾーンは未確認で、値をUTCへ変換しません。
1970年の1イベントは元LFM-2bの記載期間外で、図では2005-02-14以降に拡大しますが、集計・累積割合から除外しません。
環境変数で別の集計JSONを指定した場合は、そのファイルの規模をData Loadingの表で確認してください。

In [ ]:
temporal = temporal_rows(profile)
days = [row["day"] for row in temporal]
counts = [row["n_events"] for row in temporal]
cumulative = [row["cumulative_fraction"] for row in temporal]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(days, counts, color="#256b78", linewidth=0.8)
axes[0].set(title="Reference only: full-event temporal distribution", ylabel="Events per day")
axes[1].plot(days, cumulative, color="#256b78")
axes[1].set(ylabel="Cumulative event fraction", ylim=(0, 1.02))
plot_start = max(days[0], datetime(2005, 2, 14))
if days[-1] > plot_start:
    axes[1].set_xlim(plot_start, days[-1])
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
display(profile["train"])

## 5. Data Splitting

| 正式な保存済み分割 | 現在の状態 |
| --- | --- |
| Train / Validation / Test | 未作成 |
| 分割後の件数・時系列範囲 | 未検証 |
| 正式分割のcold-start率・ground truth件数 | 未検証 |

現カタログには再生関係がなく、ここから分割件数や正解を作りません。Notebookで分割処理を実行しません。
正式分割の保存物ができた段階で、その読込み・検証を追加します。

### 参考：保存済み分割候補の検証

以下は全量EDAで以前に調べた候補境界と集計値です。**正式な分割・採用済み比率ではありません。**
イベント比率と日付境界、Warm対象の未聴正解数を参考として確認します。
`validation` はtrain学習、`test_train_only` はtrainのみ、`test_refit_train_validation` はtrain+validationを学習側とする候補です。

cold-startの分母は各期間のdistinct user-track pairsです。理由は `cold_user → cold_item → seen → eligible` の優先順で排他的に数えています。
両方coldのペアはcold_userに入り、cold_item率は「cold_userへ分類されなかったcold itemペア」の割合です。
`novel_truth_pairs` はこの候補条件で適格な未聴正解ペア数、`users_with_novel_truth` は未聴正解を持つユーザー数です。推薦精度ではありません。
評価期間にペアがない場合、cold-start率は未定義として表・図に注記し、0へ置き換えません。

In [ ]:
display(
    {"status": "Reference candidate only; no official split artifacts", **profile["boundaries"]}
)
candidates = split_candidate_tables(profile)
display(DataFrame(candidates["events"]))
display(DataFrame(candidates["eligibility"]))
phases = candidates["eligibility"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [row["condition"] for row in phases]
positions = list(range(len(phases)))
for metric, offset, color, label in (
    ("cold_user_fraction", -0.18, "#8b507b", "Cold user (exclusive)"),
    ("cold_item_fraction", 0.18, "#c77936", "Cold item (exclusive)"),
):
    defined = [
        (position, row[metric]) for position, row in enumerate(phases) if row[metric] is not None
    ]
    axes[0].bar(
        [position + offset for position, _ in defined],
        [value for _, value in defined],
        width=0.36,
        color=color,
        label=label,
    )
undefined = [row["condition"] for row in phases if row["unique_pairs"] == 0]
if undefined:
    axes[0].text(
        0.02,
        0.97,
        "Undefined (no pairs): " + ", ".join(undefined),
        transform=axes[0].transAxes,
        va="top",
        fontsize=8,
    )
axes[0].set(
    title="Reference: cold-start pair fractions",
    ylabel="Fraction of unique pairs",
    xticks=positions,
    xticklabels=labels,
    ylim=(0, 1),
)
axes[0].legend()
axes[1].bar(labels, [row["novel_truth_pairs"] for row in phases], color="#256b78")
axes[1].set(title="Reference: eligible novel truth", ylabel="Unique user-track pairs")
for ax in axes:
    ax.tick_params(axis="x", rotation=15, labelsize=8)
fig.tight_layout()
plt.show()

## 6. Summary

- 保存済みデータの出典・版・snapshot・規模を確認し、SQL結果または合成fixtureをDataFrameへ読み込んだ。
- 件数・欠損・ID重複とタグの状態を確認した。DBモードでは公開・所属・参照・manifest・Parquetも照合する。
- EDAは曲タグの出現率・個数・重みを中心に行った。ユーザー属性・再生履歴の分析は現カタログではできない。
- 全量イベントの時刻分布・分割候補は対象規模を区別した参考資料。正式分割の件数・cold-start率・ground truthは未検証。
- 次は正式分割の条件を決定して保存物を作成し、検証結果を追加する。推薦結果は02、指標比較は03で扱う。